In [ ]:
hmap: dict = {}
hmap[128314] = "小罗"
hmap[167502] = "小算"
hmap[10583]="小鸭"

name: str = hmap[128314]
print(hmap)
hmap.pop(128314)
print(hmap)

# 遍历哈希表
for key, value in hmap.items():
    print(key, "->", value)

for key in hmap.keys():
    print(key)

for value in hmap.values():
    print(value)

#### 哈希表的简单实现
仅使用一个数组来实现哈希表。将数组中的每一个空位称为桶(bucket)，每一个桶可以储存一个键值对。查询操作通过找到`key`对应的桶，并从桶中获取`value`

通过哈希函数基于`key`定位对应的桶。

1. 通过某种哈希算法`hash()`计算得到哈希值
2. 将哈希值对桶数量`capacity`取模，从而得到该`key`对应的桶的索引`index`

下面的简单实的hash函数为$hash(key) = key$，`capacity = 100`

In [ ]:
class Pair:
    def __init__(self, key: int, val: str):
        self.key = key
        self.val = val

class ArrayHashMap:
    def __init__(self):
        self.buckets:list[Pair | None] = [None] * 100

    @staticmethod
    def hash_func(self, key: int):
        index = key % 100
        return index

    def get(self, key: int):
        index: int = self.hash_func(key)
        pair: Pair = self.buckets[index]

        if pair is None:
            return None

        return pair.val

    def put(self, key, val):
        pair = Pair(key, val)
        index = self.hash_func(key)
        self.buckets[index] = pair

    def remove(self, key):
        index = self.hash_func(key)
        self.buckets[index] = None

    def entry_set(self) -> list[Pair]:
        """获取所有键值对"""
        result: list[Pair] = []
        for pair in self.buckets:
            if pair is not None:
                result.append(pair)
        return result

    def key_set(self) -> list[int]:
        result: list[int] = []
        for pair in self.buckets:
            if pair is not None:
                result.append(pair.key)

        return result

    def value_set(self) -> list[str]:
        result: list[str] = []
        for pair in self.buckets:
            if pair is not None:
                result.append(pair.val)

        return result

    def print(self):
        for pair in self.buckets:
            if pair is not None:
                print(pair.key, "->", pair.val)

在上述代码中使用到了静态方法，其作用是避免为了调用类中不需要使用到`self`中属性的函数而创建多余的类的实例，从而避免了浪费cpu以及内存

#### 哈希冲突与扩容
由于哈希函数一般不是双射，**理论上存在多个输入对应相同输出**的情况，这种情况被称为哈希冲突。

可以通过扩容哈希表来避免哈希冲突。

负载因子（loadfactor）是哈希表的一个重要概念，其定义为哈希表的元素数量除以桶数量，用于衡量哈希冲突的严重程度，也常作为**哈希表扩容的触发条件**。例如在Java中，当负载因子超过0.75时，系统会将哈希表扩容至原先的2倍。


#### 为了解决哈希冲突的哈希表结构改良方法
1. 链式地址
链式地址（separate-chaining）将单个元素转换为链表，将键值对作为链表节点，将所有发生冲突的键值对都存储在同一链表中。
以下的代码给出了链式地址哈希表的简单实现，注意：
- 使用列表（动态数组）代替链表简化代码，在这种情况下，哈希表（数组）包含多个桶，每一个桶都是一个列表。
- 当负载因子超过$\frac{2}{3}$时，将哈希表扩容至原来的$2$倍

In [ ]:
class HashMapChaining:
    def __init__(self):
        self.size = 0
        self.capacity = 0
        self.load_thres = 2.0 / 3.0 # 触发扩容的负载因子
        self.extend_ratio = 2
        self.buckets = [[] for _ in range(self.capacity)]

    def hash_func(self, key: int) -> int:
        return key % self.capacity

    def load_factor(self) -> float:
        return self.size / self.capacity

    def get(self, key: int) -> str|None:
        index = self.hash_func(key)
        bucket = self.buckets[index]
        # 遍历桶，若找到key，返回对应的val
        for pair in bucket:
            if pair.key == key:
                return pair.val

        # 如果没有找到
        return None

    def put(self, key, val):
        if self.load_factor() > self.load_thres:
            self.extend()
        index = self.hash_func(key)
        bucket = self.buckets[index]
        # 遍历桶，若遇到指定key，则更新对应val并返回
        for pair in bucket:
            if pair.key == key:
                pair.val = val

        # 若无该key，则将键值对添加到尾部
        pair = Pair(key, val)
        bucket.append(pair)
        self.size += 1

    def remove(self, key):
        index = self.hash_func(key)
        bucket = self.buckets[index]
        for pair in bucket:
            if pair.key == key:
                bucket.remove(pair)
                self.size -= 1
                break

    def extend(self):
        # 扩容哈希表
        # 暂存哈希表
        buckets = self.buckets
        self.capacity *= self.extend_ratio
        self.buckets = [[] for _ in range(self.capacity)]
        self.size = 0
        for bucket in buckets:
            for pair in bucket:
                self.put(pair.key, pair.val)

    def print(self):
        for bucket in self.buckets:
            res = []
            for pair in bucket:
                res.append(str(pair.key) + "->" + pair.val)
            print(res)

2. 开放寻址
以线性探测为例，介绍开放寻址哈希表的工作机制
线性探测采取固定补偿的线性搜索来进行探测
- 插入元素：通过哈希函数计算桶索引，若桶内已有元素，测从冲突位置向后线性遍历（步长通常为1），知道找到空桶，将元素插入其中
- 查找元素：如果发现哈希冲突，则使用相同步长向后进行线性遍历，直到找到对应元素，返回`value`即可；若遇到空桶，说明元素不在哈希表中，返回`None`

注意：不能在开放寻址哈希表直接删除元素，改为懒删除：：它不直接从哈希表中移除元素，而是利用一个常量
`TOMBSTONE` 来标记这个桶。在该机制下，`None` 和`TOMBSTONE` 都代表空桶，都可以放置键值对。但不同的是，线性探测到`TOMBSTONE` 时应该继续遍历，因为其之下可能还存在键值对。

然而，懒删除可能会加速哈希表的性能退化，随着`TOMBSTONE`的增加，搜索时间也会增加。

In [ ]:
class HashMapOpenAddressing:
    def __init__(self):
        self.size = 0
        self.capacity = 4
        self.load_thres =2.0 / 3.0
        self.extend_ratio = 2
        self.buckets = [None] * self.capacity
        self.TOMBSTONE = Pair(-1, "-1")

    def hash_func(self, key):
        return key % self.capacity

    def load_factor(self):
        return self.size / self.capacity

    def find_bucket(self, key):
        """查询key所对应的桶索引"""
        index = self.hash_func(key)
        first_tombstone = -1
        # 线性探测，当遇到空桶时跳出
        while self.buckets[index] is not None:
            # 如果找到了key，返回对应的index
            if self.buckets[index].key == key:
                # 若之前遇到了删除标记，则将键值对移动该索引处
                if first_tombstone != -1:
                    self.buckets[first_tombstone] = self.buckets[index]
                    self.buckets[index] = self.TOMBSTONE
                    return first_tombstone
                # 若没遇到，则直接返回对应的index
                return index

            # 遇到第一个墓碑，则记录下来，用来记录这一个key的桶索引
            if first_tombstone == -1 and self.buckets[index] is self.TOMBSTONE:
                first_tombstone = index

            index = (index + 1) % self.capacity
        # 如果直接遇到了空桶，返回index；如果非空，那么进行线性遍历，步长为1，如果遇到墓碑则记录首个墓碑的index
        return index if first_tombstone == -1 else first_tombstone

    def get(self, key: int):
        index = self.find_bucket(key)
        if self.buckets[index] not in [None, self.TOMBSTONE]:
            return self.buckets[index].val
        return None

    def put(self, key, val):
        # 确保在桶中有`None`可供插入
        if self.load_factor() > self.load_thres:
            self.extend()
        index = self.find_bucket(key)
        # 若找到键值对，覆盖val并返回
        if self.buckets[index] not in [None, self.TOMBSTONE]:
            self.buckets[index].val = val
            return
        # 若键值对不存在，则添加该键值对
        self.buckets[index] =  Pair(key, val)
        self.size += 1

    def remove(self, key):
        index = self.find_bucket(key)

    def extend(self):
        buckets_tmp = self.buckets
        self.capacity *= self.extend_ratio
        self.buckets = [None] * self.capacity
        self.size = 0
        for pair in buckets_tmp:
            if pair not in [None, self.TOMBSTONE]:
                self.put(pair.key, pair.val)
        # 为什么在扩容哈希表的时候不需要复制墓碑到新的哈希表中？
        # 墓碑唯一的作用时在当前容量下对于对于已经删除的元素的记录，扩容至后，一切从头开始对于所有其他操作没有影响